# Comprobación de la instalación

Este cuaderno tiene como objetivo facilitar la comprobación de la instalación de PySpark.

## Comprobar la disponibilidad del paquete `pyspark`

Para comprobarlo, primero vamos a ver si podemos importar PySpark:

In [1]:
import pyspark

## Comprobar la versión instalada

A continuación, vamos a comprobar la versión de PySpark que estamos usando:

In [2]:
pyspark.__version__

'3.5.8'

## Iniciar una sesión Spark

Gracias a que definimos el servicio `jupyter` con la variable de entorno:

```yml
environment:
  - SPARK_MASTER=spark://spark-master:7077
```

podemos ahorrarnos el parámetro `.master` en nuestra llamada:

```python
spark = SparkSession.builder \
    .master('spark-master:7077')
    .appName('test') \
    .getOrCreate()
```

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 19:40:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.8


### Prueba de funcionamiento con un fichero CSV

Para terminar nuestra prueba, vamos a probar a hacer una prueba mínima pero realista usando el fichero de zonas de los taxis de Nueva York.

In [4]:
# Eliminamos ficheros de ejecuciones anteriores, si los hubiese
!rm -rf zones

# Descargamos el fichero taxi_zone_lookup.csv
!wget https://raw.githubusercontent.com/DataTalksClub/data-engineering-zoomcamp/refs/heads/main/04-analytics-engineering/taxi_rides_ny/seeds/taxi_zone_lookup.csv

--2026-02-24 19:40:10--  https://raw.githubusercontent.com/DataTalksClub/data-engineering-zoomcamp/refs/heads/main/04-analytics-engineering/taxi_rides_ny/seeds/taxi_zone_lookup.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12056 (12K) [text/plain]
Saving to: ‘taxi_zone_lookup.csv.22’

taxi_zone_lookup.cs 100%[===================>]  11.77K  --.-KB/s    in 0.002s  

2026-02-24 19:40:11 (4.78 MB/s) - ‘taxi_zone_lookup.csv.22’ saved [12056/12056]



In [5]:
df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

df.show()
df.write.parquet('zones')

+----------+-------------+--------------------+------------+
|locationid|      borough|                zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [6]:
!ls -lh zones

total 8.0K
-rw-r--r-- 1 root root    0 Feb 24 19:40 _SUCCESS
-rw-r--r-- 1 root root 5.8K Feb 24 19:40 part-00000-2d0021f5-367f-46c7-9089-0d0751a4f128-c000.snappy.parquet
